# Lecture 3 · Notebook 2 — A fixed-window neural language model (Bengio et al., 2003)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture03/lecture03_2_fixed_window_neural_lm.ipynb)

**Companion to [Chapter 6 of the video lecture](https://nlp-course-cuhksz.github.io/lecture03.html#p6-s90)** (slides 84–95). N-gram models suffer from **sparsity** and
**storage**. The fixed-window neural LM (slides 90–91) fixes both: it embeds the last few words, concatenates the embeddings, applies a
hidden layer, and predicts the next word with a softmax over the vocabulary. In this notebook you will:

1. build it in about 15 lines of PyTorch,
2. train it on the same *tiny Shakespeare* split as Notebook 1 and compare **perplexity** with an interpolated trigram model,
3. sample from it with different **temperatures**, and
4. look at the **word embeddings** it learned as a by-product (slide 93).

Runs on a free CPU runtime in about 5–8 minutes, most of it training (a GPU makes it faster but is not needed).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, time
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

Same data preparation as Notebook 1: lowercase, words and punctuation, one sentence per line, rare words → `<unk>`, 10% test lines.

In [ ]:
import os, re, math, random, urllib.request
from collections import Counter, defaultdict

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
raw = open("tinyshakespeare.txt", encoding="utf-8").read()

def tokenize(line):
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[^\sa-z]", line.lower())

lines = [tokenize(l) for l in raw.split("\n")]
lines = [l for l in lines if l]                      # every non-empty line is one "sentence"
random.Random(0).shuffle(lines)
n_test, n_dev = len(lines) // 10, len(lines) // 20
test_lines, dev_lines, train_lines = lines[:n_test], lines[n_test:n_test + n_dev], lines[n_test + n_dev:]

# vocabulary: words seen at least twice in training; everything else becomes <unk> (slide 45)
wc = Counter(w for l in train_lines for w in l)
vocab = {w for w, c in wc.items() if c >= 2} | {"<unk>", "</s>"}
def unkify(l):
    return [w if w in vocab else "<unk>" for w in l]
train = [unkify(l) for l in train_lines]
test = [unkify(l) for l in test_lines]
dev = [unkify(l) for l in dev_lines]       # held-out lines for tuning (never used for counting or training)
print(f"{len(train)} training lines, {sum(map(len, train)):,} training tokens, {len(dev)} dev lines, {len(test)} test lines")
print(f"vocabulary: {len(vocab):,} types (incl. <unk>, </s>)")
print("example line:", train[0])

## 1 · Turn lines into (context → next word) examples
With a window of 3, the model sees `w[t-3], w[t-2], w[t-1]` and predicts `w[t]`, exactly like a 4-gram model. `<s>` pads the start.

In [ ]:
WINDOW = 3
itos = ["<s>"] + sorted(vocab)
stoi = {w: i for i, w in enumerate(itos)}

def examples(lines):
    X, Y = [], []
    for l in lines:
        ids = [stoi["<s>"]] * WINDOW + [stoi[w] for w in l] + [stoi["</s>"]]
        for t in range(WINDOW, len(ids)):
            X.append(ids[t - WINDOW:t]); Y.append(ids[t])
    return torch.tensor(X), torch.tensor(Y)

Xtr, Ytr = examples(train)
Xte, Yte = examples(test)
print("train examples:", tuple(Xtr.shape), " test examples:", tuple(Xte.shape))
print("one example:", [itos[i] for i in Xtr[5]], "->", itos[Ytr[5]])

## 2 · The model (slide 90)
$e = [E x_{t-3}; E x_{t-2}; E x_{t-1}]$, $\;h = f(W e + b_1)$, $\;\hat y = \mathrm{softmax}(U h + b_2)$.

In [ ]:
class FixedWindowLM(nn.Module):
    def __init__(self, V, window=WINDOW, d_emb=64, d_hid=256, dropout=0.3):
        super().__init__()
        self.E = nn.Embedding(V, d_emb)                 # word embeddings, learned as a by-product
        self.W = nn.Linear(window * d_emb, d_hid)       # hidden layer over the concatenated window
        self.U = nn.Linear(d_hid, V)                    # output layer: one score per vocabulary word
        self.drop = nn.Dropout(dropout)

    def forward(self, x):                               # x: [batch, window] word ids
        e = self.E(x).flatten(1)                        # [batch, window * d_emb]
        h = torch.tanh(self.W(self.drop(e)))
        return self.U(self.drop(h))                     # logits; softmax is inside the loss

model = FixedWindowLM(len(itos)).to(device)
print(model)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")

## 3 · Train with cross-entropy and track test perplexity
Cross-entropy is the average negative log-probability of the true next word, so **perplexity = exp(cross-entropy)** — the same
quantity as in Notebook 1.

In [ ]:
@torch.no_grad()
def test_ppl(m):
    m.eval()
    nll = 0.0
    for i in range(0, len(Xte), 4096):
        x, y = Xte[i:i + 4096].to(device), Yte[i:i + 4096].to(device)
        nll += F.cross_entropy(m(x), y, reduction="sum").item()
    m.train()
    return math.exp(nll / len(Xte))

opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
EPOCHS, BS = 4, 512
print(f"before training: test perplexity {test_ppl(model):.0f} (about the vocabulary size: a uniform guess)")
for ep in range(EPOCHS):
    t0, perm = time.time(), torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), BS):
        idx = perm[i:i + BS]
        loss = F.cross_entropy(model(Xtr[idx].to(device)), Ytr[idx].to(device))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {ep + 1}: train loss {loss.item():.2f}  test perplexity {test_ppl(model):.1f}  ({time.time() - t0:.0f}s)")

## 4 · Compare with an interpolated trigram model
The same interpolated trigram as in Notebook 1 (fixed λ's for brevity), on the same test lines and vocabulary.

In [ ]:
BOS, EOS = "<s>", "</s>"
def build(lines, n):
    t = defaultdict(Counter)
    for l in lines:
        p = [BOS] * (n - 1) + l + [EOS]
        for i in range(len(p) - n + 1):
            t[tuple(p[i:i + n - 1])][p[i + n - 1]] += 1
    for c in t.values():
        c.total = sum(c.values())
    return t
T = {n: build(train, n) for n in (1, 2, 3)}
Vp = len(vocab)

def p_mle(n, ctx, w):
    c = T[n].get(ctx)
    return c[w] / c.total if c else 0.0

def trigram_ppl(lams=(0.3, 0.5, 0.2)):
    tot, N = 0.0, 0
    for l in test:
        p = [BOS, BOS] + l + [EOS]
        for i in range(2, len(p)):
            ctx, w = tuple(p[i - 2:i]), p[i]
            p1 = (T[1][()][w] + 1) / (T[1][()].total + Vp)
            tot += math.log(lams[0] * p_mle(3, ctx, w) + lams[1] * p_mle(2, ctx[1:], w) + lams[2] * p1)
            N += 1
    return math.exp(-tot / N)

print(f"interpolated trigram : {trigram_ppl():.1f}")
print(f"fixed-window neural  : {test_ppl(model):.1f}")

In our test run the neural LM reached a perplexity of about 107 against about 123 for this trigram (about 118 with the tuned λ's of
Notebook 1): a modest win on a corpus this small, and it depends on the window, the model size and the epochs (try the exercises).
The neural model's advantages from slide 91 still show: it never assigns zero probability, it stores
**no n-gram table** (its size does not grow with the corpus), and similar words share statistics through their embeddings. Its
remaining weakness is the **fixed window**: anything more than 3 words back is invisible, which is why RNNs and then Transformers came next.

## 5 · Sampling with a temperature
Each step is one spin of a wheel with |V| slices. Dividing the logits by a temperature *T* reshapes the wheel: *T* < 1 makes it greedier,
*T* > 1 flatter.

In [ ]:
@torch.no_grad()
def sample(prompt="", T=1.0, max_len=30, seed=0):
    g = torch.Generator().manual_seed(seed)
    model.eval()
    ctx = [stoi["<s>"]] * WINDOW + [stoi.get(w, stoi["<unk>"]) for w in tokenize(prompt)]
    out = []
    for _ in range(max_len):
        logits = model(torch.tensor([ctx[-WINDOW:]], device=device))[0].cpu() / T
        w = torch.multinomial(F.softmax(logits, -1), 1, generator=g).item()
        if itos[w] == "</s>":
            break
        out.append(itos[w]); ctx.append(w)
    model.train()
    return " ".join(tokenize(prompt) + out)

for T in [0.5, 1.0, 1.5]:
    for seed in (1, 2):
        print(f"T={T}:", sample("my lord", T=T, seed=seed))

## 6 · Word vectors as a by-product (slides 93–95)
To predict the next word, the model learned **two** vectors per word, just like word2vec's *v* and *u* (slide 95): an input embedding
(a row of `E`) and an output vector (a row of `U`, the word's weights in the softmax). Words that are used in similar contexts should end
up close together. We compare nearest neighbours among the 1,500 most frequent words.

In [ ]:
freq = Counter(w for l in train for w in l)
common = [w for w, _ in freq.most_common(1500)]
idx = torch.tensor([stoi[w] for w in common])

def neighbours(M, w, k=6):
    X = F.normalize(M.detach().cpu()[idx], dim=1)
    sims = X @ X[common.index(w)]
    best = sims.topk(k + 1).indices.tolist()[1:]
    return ", ".join(f"{common[i]} ({sims[i]:.2f})" for i in best)

for name, M in [("output vectors U", model.U.weight), ("input embeddings E", model.E.weight)]:
    print("==", name)
    for w in ["king", "father", "thee", "he", "go", "night", "good"]:
        print(f"   {w:7s} -> {neighbours(M, w)}")

On this small corpus the **output vectors** are already meaningful (titles near titles, relatives near relatives, pronouns near pronouns,
*night* near *day*), while the input embeddings are still noisy after a few epochs. Word2vec (slide 95) gets much better vectors by
training a simpler model on far more text.

## Exercises
1. Change `WINDOW` to 1, 2 and 5. How does test perplexity change? Relate this to the n-gram order.
2. Remove dropout (`dropout=0.0`) and train for 8 epochs. Watch the train loss and the test perplexity. What happens, and why?
3. Mix the models: average the neural LM's probability with the interpolated trigram's probability for each test token
   (0.5/0.5). Is the mixture better than both?
4. Count parameters: how many would a full 4-gram **table** need (V⁴ entries) versus this model? (Slide 88.)

**Next:** [Notebook 3 — sampling from GPT-2 and filling BERT's blanks](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture03/lecture03_3_sampling_gpt2_and_bert.ipynb) · **Back to the [video lecture](https://nlp-course-cuhksz.github.io/lecture03.html)**